In [ ]:
import pandas as pd
import numpy as np

def analyze_all_sv_languages(data):
    """
    Analyze ALL SV languages without sampling.
    Returns languages aligned with or deviating from the expected order pattern.
    """
    # Convert to strings
    for col in ['GB130', 'GB065', 'GB193']:
        if col in data.columns:
            data[col] = data[col].astype(str)
    
    # Filter for rigid SV languages with rigid possessor and adjective order
    has_sv = data['GB130'] == '1'  # Only rigid SV
    has_poss = data['GB065'].isin(['1', '2'])  # Only rigid possessive order
    has_adj = data['GB193'].isin(['1', '2'])  # Only rigid adjective order
    
    sv_languages = data[has_sv & has_poss & has_adj].copy()
    
    print(f"\n{'='*70}")
    print("FULL DATASET ANALYSIS - NO SAMPLING")
    print(f"{'='*70}")
    print(f"\nTotal languages in dataset: {len(data)}")
    print(f"Languages with GB130='1' (rigid SV): {(data['GB130']=='1').sum()}")
    print(f"Languages with rigid SV + rigid possessive + rigid adjective: {len(sv_languages)}")
    
    if len(sv_languages) == 0:
        print("\nNo languages found matching criteria!")
        return None
    
    # Define alignment-friendly patterns:
    # Pattern 1: GEN-N (GB065='1') - possessor before noun
    # Pattern 2: N-GEN AND N-ADJ (GB065='2' AND GB193='2') - both after noun
    
    has_gen_n = sv_languages['GB065'] == '1'
    has_n_gen_and_n_adj = (sv_languages['GB065'] == '2') & (sv_languages['GB193'] == '2')
    
    matches_alignment = has_gen_n | has_n_gen_and_n_adj
    
    # Separate into aligned vs mixed order
    aligned_langs = sv_languages[matches_alignment].copy()
    mixed_order_langs = sv_languages[~matches_alignment].copy()
    
    # Calculate statistics
    num_aligned = len(aligned_langs)
    num_mixed = len(mixed_order_langs)
    total = len(sv_languages)
    percentage_aligned = (num_aligned / total * 100) if total > 0 else 0
    
    print(f"\n{'='*70}")
    print("RESULTS")
    print(f"{'='*70}")
    print(f"\nTotal rigid SV languages analyzed: {total}")
    print(f"Languages matching alignment pattern: {num_aligned} ({percentage_aligned:.2f}%)")
    print(f"Languages with mixed order: {num_mixed} ({100-percentage_aligned:.2f}%)")
    
    # Break down by order pattern
    print(f"\n{'='*70}")
    print("BREAKDOWN BY PATTERN TYPE")
    print(f"{'='*70}")
    
    # Pattern 1: GEN-N (any adjective order)
    gen_n_adjn = sv_languages[(sv_languages['GB065'] == '1') & (sv_languages['GB193'] == '1')]
    gen_n_nadj = sv_languages[(sv_languages['GB065'] == '1') & (sv_languages['GB193'] == '2')]
    
    # Pattern 2: N-GEN AND N-ADJ
    n_gen_n_adj = sv_languages[(sv_languages['GB065'] == '2') & (sv_languages['GB193'] == '2')]
    
    # Mixed: N-GEN AND ADJ-N
    n_gen_adjn = sv_languages[(sv_languages['GB065'] == '2') & (sv_languages['GB193'] == '1')]
    
    print(f"\nALIGNED PATTERNS:")
    print(f"  GEN-N + ADJ-N: {len(gen_n_adjn)} languages ({len(gen_n_adjn)/total*100:.1f}%)")
    print(f"  GEN-N + N-ADJ: {len(gen_n_nadj)} languages ({len(gen_n_nadj)/total*100:.1f}%)")
    print(f"  N-GEN + N-ADJ: {len(n_gen_n_adj)} languages ({len(n_gen_n_adj)/total*100:.1f}%)")
    print(f"\nMIXED PATTERN:")
    print(f"  N-GEN + ADJ-N: {len(n_gen_adjn)} languages ({len(n_gen_adjn)/total*100:.1f}%)")
    
    return {
        'total': total,
        'aligned': aligned_langs,
        'mixed': mixed_order_langs,
        'gen_n_adjn': gen_n_adjn,
        'gen_n_nadj': gen_n_nadj,
        'n_gen_n_adj': n_gen_n_adj,
        'n_gen_adjn': n_gen_adjn
    }

def print_language_lists(results, metadata):
    """Print detailed lists of languages in each category"""
    
    print(f"\n\n{'='*70}")
    print("MIXED ORDER LANGUAGES (N-GEN + ADJ-N)")
    print(f"{'='*70}")
    
    if len(results['mixed']) == 0:
        print("\nNo mixed-order languages found!")
    else:
        print(f"\nTotal: {len(results['mixed'])} languages\n")
        
        for idx, (_, row) in enumerate(results['mixed'].iterrows(), 1):
            lang_name = row['Language']
            family = metadata.get(lang_name, {}).get('family', 'Unknown')
            macroarea = metadata.get(lang_name, {}).get('macroarea', 'Unknown')
            
            print(f"{idx}. {lang_name}")
            print(f"   Family: {family}")
            print(f"   Macroarea: {macroarea}")
            print(f"   Pattern: SV + N-GEN + ADJ-N")
            print()
    
    print(f"\n{'='*70}")
    print("ALIGNED LANGUAGES - BY PATTERN TYPE")
    print(f"{'='*70}")
    
    # Print first 20 of each aligned pattern type
    patterns = [
        ('GEN-N + ADJ-N', results['gen_n_adjn']),
        ('GEN-N + N-ADJ', results['gen_n_nadj']),
        ('N-GEN + N-ADJ', results['n_gen_n_adj'])
    ]
    
    for pattern_name, pattern_langs in patterns:
        print(f"\n{pattern_name}: {len(pattern_langs)} languages")
        print("-" * 70)
        
        if len(pattern_langs) == 0:
            print("None found\n")
            continue
        
        # Show first 20
        display_count = min(20, len(pattern_langs))
        print(f"Showing first {display_count} of {len(pattern_langs)}:\n")
        
        for idx, (_, row) in enumerate(pattern_langs.head(20).iterrows(), 1):
            lang_name = row['Language']
            family = metadata.get(lang_name, {}).get('family', 'Unknown')
            macroarea = metadata.get(lang_name, {}).get('macroarea', 'Unknown')
            
            print(f"{idx}. {lang_name} ({family}, {macroarea})")
        
        if len(pattern_langs) > 20:
            print(f"\n... and {len(pattern_langs) - 20} more")
        print()

def save_full_lists(results, metadata):
    """Save complete lists to CSV files"""
    
    # Save mixed-order languages
    if len(results['mixed']) > 0:
        mixed_export = results['mixed'][['Language', 'GB130', 'GB065', 'GB193']].copy()
        mixed_export['Family'] = mixed_export['Language'].map(
            lambda x: metadata.get(x, {}).get('family', 'Unknown')
        )
        mixed_export['Macroarea'] = mixed_export['Language'].map(
            lambda x: metadata.get(x, {}).get('macroarea', 'Unknown')
        )
        mixed_export['Pattern'] = 'N-GEN + ADJ-N (mixed order)'
        mixed_export.to_csv('mixed_order_languages.csv', index=False)
        print(f"\nSaved {len(mixed_export)} mixed-order languages to 'mixed_order_languages.csv'")
    
    # Save all aligned languages by pattern
    all_aligned = []
    
    for pattern_name, pattern_data in [
        ('GEN-N + ADJ-N', results['gen_n_adjn']),
        ('GEN-N + N-ADJ', results['gen_n_nadj']),
        ('N-GEN + N-ADJ', results['n_gen_n_adj'])
    ]:
        if len(pattern_data) > 0:
            pattern_export = pattern_data[['Language', 'GB130', 'GB065', 'GB193']].copy()
            pattern_export['Family'] = pattern_export['Language'].map(
                lambda x: metadata.get(x, {}).get('family', 'Unknown')
            )
            pattern_export['Macroarea'] = pattern_export['Language'].map(
                lambda x: metadata.get(x, {}).get('macroarea', 'Unknown')
            )
            pattern_export['Pattern'] = pattern_name
            all_aligned.append(pattern_export)
    
    if all_aligned:
        aligned_export = pd.concat(all_aligned, ignore_index=True)
        aligned_export.to_csv('aligned_languages.csv', index=False)
        print(f"Saved {len(aligned_export)} aligned languages to 'aligned_languages.csv'")

def main():
    # Load data
    print("Loading data...")
    grambank = pd.read_csv('grambank_sane_format.csv')
    languages = pd.read_csv('languages1.csv')
    
    # Create metadata dictionary
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family_name']
            }
    
    # Add metadata to grambank
    grambank['Macroarea'] = grambank['Language'].map(
        lambda x: metadata.get(x, {}).get('macroarea')
    )
    grambank['Family'] = grambank['Language'].map(
        lambda x: metadata.get(x, {}).get('family')
    )
    
    # Analyze all languages
    results = analyze_all_sv_languages(grambank)
    
    if results is None:
        return
    
    # Print language lists
    print_language_lists(results, metadata)
    
    # Save to CSV
    save_full_lists(results, metadata)
    
    print(f"\n{'='*70}")
    print("ANALYSIS COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Loading data...

FULL DATASET ANALYSIS - NO SAMPLING

Total languages in dataset: 2467
Languages with GB130='1' (rigid SV): 1827
Languages with rigid SV + rigid possessive + rigid adjective: 1167

RESULTS

Total rigid SV languages analyzed: 1167
Languages matching harmonic pattern: 1114 (95.46%)
Languages NOT matching (disharmonic): 53 (4.54%)

BREAKDOWN BY PATTERN TYPE

HARMONIC PATTERNS:
  GEN-N + ADJ-N: 278 languages (23.8%)
  GEN-N + N-ADJ: 417 languages (35.7%)
  N-GEN + N-ADJ: 419 languages (35.9%)

DISHARMONIC PATTERN:
  N-GEN + ADJ-N: 53 languages (4.5%)


DISHARMONIC LANGUAGES (N-GEN + ADJ-N)

Total: 53 languages

1. Aja (South Sudan)
   Family: Kresh-Aja
   Macroarea: Africa
   Pattern: SV + N-GEN + ADJ-N

2. Ajyíninka Apurucayali
   Family: Arawakan
   Macroarea: South America
   Pattern: SV + N-GEN + ADJ-N

3. Apma
   Family: Austronesian
   Macroarea: Papunesia
   Pattern: SV + N-GEN + ADJ-N

4. Arigidi
   Family: Atlantic-Congo
   Macroarea: Africa
   Pattern: SV + N-GEN 